# 03. Data Preprocessing - Automobile Loan Default Prediction

**Purpose:** Convert the documented raw-data problems into a reproducible, leakage-safe dataset preparation workflow.

This notebook begins from the evidence recorded in `01_data_understanding.ipynb` on the `main` branch:

- 121,856 loan applications and 40 columns
- `Default` is an imbalanced binary target
- nine numerical variables are stored as text because of corrupted symbols
- 33 columns contain missing values
- no duplicate rows or duplicate IDs were found
- `ID` is an identifier, not a predictive characteristic

We re-check those claims before cleaning. We do not assume that every interpretation in the earlier notebook is automatically correct.

## 1. Preprocessing principles

The order of operations matters:

1. Preserve the raw data.
2. Apply deterministic corrections based on domain validity, not learned statistics.
3. Separate identifiers, predictors, and target.
4. Split into training and test data using target stratification.
5. Fit imputation, encoding, and scaling on training data only.
6. Reuse the fitted preprocessing object to transform test or future data.

This prevents test-set information from influencing the values learned during preprocessing.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 1200)

RANDOM_STATE = 42
TEST_SIZE = 0.20

## 2. Load and preserve the raw dataset

`low_memory=False` asks pandas to infer a consistent type for each complete column. It does not clean any values. We use the name `raw_df` and create a deep copy before making changes.

In [2]:
project_root = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
raw_data_path = project_root / 'data' / 'raw' / 'Train_Dataset.csv'

if not raw_data_path.exists():
    raise FileNotFoundError(f'Raw dataset not found: {raw_data_path}')

raw_df = pd.read_csv(raw_data_path, low_memory=False)
df = raw_df.copy(deep=True)

print(f'Loaded: {raw_data_path}')
print(f'Raw shape: {raw_df.shape}')
print(f'Working copy created: {df is not raw_df}')

Loaded: C:\Users\yasan\Downloads\Sem 2 - 3rd Year\FDM\FDM-AutoMobile_Loan_Default_Modelling\data\raw\Train_Dataset.csv
Raw shape: (121856, 40)
Working copy created: True


## 3. Reproduce the Data Understanding controls

A notebook should fail early if a different dataset version is used. These assertions are not cleaning; they verify that the input matches the dataset studied by the team.

In [3]:
baseline_checks = pd.Series({
    'rows': len(raw_df),
    'columns': raw_df.shape[1],
    'columns_with_missing_values': int(raw_df.isna().any().sum()),
    'duplicate_rows': int(raw_df.duplicated().sum()),
    'duplicate_ids': int(raw_df['ID'].duplicated().sum()),
    'missing_ids': int(raw_df['ID'].isna().sum()),
    'missing_target_values': int(raw_df['Default'].isna().sum()),
}, name='observed_value')

display(baseline_checks.to_frame())

assert raw_df.shape == (121856, 40), 'Unexpected dataset shape'
assert baseline_checks['duplicate_rows'] == 0, 'Duplicate rows require investigation'
assert baseline_checks['duplicate_ids'] == 0, 'ID is not unique'
assert baseline_checks['missing_ids'] == 0, 'ID contains missing values'
assert baseline_checks['missing_target_values'] == 0, 'Target contains missing values'
assert set(raw_df['Default'].unique()) == {0, 1}, 'Target is not binary'

,observed_value
rows,121856
columns,40
columns_with_missing_values,33
duplicate_rows,0
duplicate_ids,0
missing_ids,0
missing_target_values,0


In [4]:
target_distribution = pd.DataFrame({
    'count': raw_df['Default'].value_counts().sort_index(),
    'percentage': (raw_df['Default'].value_counts(normalize=True).sort_index() * 100).round(3),
})
target_distribution.index = ['Non-default (0)', 'Default (1)']
target_distribution

,count,percentage
Non-default (0),112011,91.921
Default (1),9845,8.079


**Decision:** No duplicate removal is needed. The approximately 8% positive class requires a stratified split. Accuracy alone will not be a sufficient model metric.

## 4. Repair numeric columns stored as text

The upstream notebook identified nine conceptually numerical columns containing symbols such as `$`, `x`, `@`, `#`, `&`, and `#VALUE!`.

For each column we distinguish:

- values already missing in the CSV; and
- non-missing values that cannot be parsed as numbers.

`errors='coerce'` changes invalid symbols to `NaN`. This is justified because the symbols are not valid measurements. It does not decide how those missing values will later be imputed.

In [5]:
numeric_text_columns = [
    'Client_Income', 'Credit_Amount', 'Loan_Annuity',
    'Population_Region_Relative', 'Age_Days', 'Employed_Days',
    'Registration_Days', 'ID_Days', 'Score_Source_3'
]

numeric_conversion_records = []

for column in numeric_text_columns:
    original = df[column]
    converted = pd.to_numeric(original, errors='coerce')
    corrupted_mask = original.notna() & converted.isna()

    numeric_conversion_records.append({
        'column': column,
        'original_missing': int(original.isna().sum()),
        'corrupted_count': int(corrupted_mask.sum()),
        'corrupted_values': original[corrupted_mask].value_counts().to_dict(),
        'missing_after_conversion': int(converted.isna().sum()),
    })

    df[column] = converted

numeric_conversion_summary = pd.DataFrame(numeric_conversion_records).set_index('column')
display(numeric_conversion_summary)
print('Total corrupted numeric entries converted to NaN:',
      numeric_conversion_summary['corrupted_count'].sum())

assert numeric_conversion_summary['corrupted_count'].sum() == 114
assert all(pd.api.types.is_numeric_dtype(df[column]) for column in numeric_text_columns)

,original_missing,corrupted_count,corrupted_values,missing_after_conversion
column,,,,
Client_Income,3607,15,{'$': 15},3622
Credit_Amount,3632,5,{'$': 5},3637
Loan_Annuity,4812,14,"{'#VALUE!': 13, '$': 1}",4826
Population_Region_Relative,4857,11,"{'@': 6, '#': 5}",4868
Age_Days,3600,17,{'x': 17},3617
Employed_Days,3649,17,{'x': 17},3666
Registration_Days,3614,17,{'x': 17},3631
ID_Days,5968,17,{'x': 17},5985
Score_Source_3,26921,1,{'&': 1},26922


Total corrupted numeric entries converted to NaN: 114


## 5. Inspect categorical values before handling possible placeholders

The Data Understanding notebook identified which columns are categorical, but it did not list every observed category. We first produce a frequency table for **all** categorical columns. Only after seeing the actual values do we investigate unexpected codes such as `XNA` and `##`. This prevents us from selecting placeholder values without showing how they were discovered.

In [ ]:
categorical_columns_before_cleaning = df.select_dtypes(
    include=['object', 'string']
).columns.tolist()

categorical_frequency_tables = []

for column in categorical_columns_before_cleaning:
    counts = df[column].value_counts(dropna=False)
    frequency_table = pd.DataFrame({
        'column': column,
        'observed_value': counts.index,
        'count': counts.values,
        'percentage': (counts.values / len(df) * 100).round(3),
    })
    categorical_frequency_tables.append(frequency_table)

categorical_value_audit = pd.concat(
    categorical_frequency_tables, ignore_index=True 
)

print('Categorical columns inspected:', len(categorical_columns_before_cleaning))
display(categorical_value_audit)

: 

The complete frequency table reveals `XNA` in `Client_Gender` and `Type_Organization`, and `##` in `Accompany_Client`. These values are now investigated using their frequency, expected domain, and relationships with other variables. The string `XNA` alone does not prove a specific meaning.

In [ ]:
placeholder_counts_before = pd.Series({
    'Client_Gender = XNA': int(df['Client_Gender'].eq('XNA').sum()),
    'Accompany_Client = ##': int(df['Accompany_Client'].eq('##').sum()),
    'Type_Organization = XNA': int(df['Type_Organization'].eq('XNA').sum()),
}, name='count')

display(placeholder_counts_before.to_frame())
display(pd.crosstab(df['Client_Income_Type'], df['Type_Organization'].eq('XNA')))

The cross-tabulation shows that `Type_Organization = XNA` is overwhelmingly associated with retired applicants. Its occurrence is systematic and potentially informative, but this pattern does not prove that `XNA` means *not applicable*. We therefore retain the original `XNA` label as its own category. The 3 gender `XNA` values and 12 accompanying-client `##` values violate the expected values of their respective fields, so they are converted to missing values.

In [ ]:
    df['Client_Gender'] = df['Client_Gender'].replace('XNA', np.nan)
    df['Accompany_Client'] = df['Accompany_Client'].replace('##', np.nan)

    assert df['Client_Gender'].eq('XNA').sum() == 0
    assert df['Accompany_Client'].eq('##').sum() == 0
    assert df['Type_Organization'].eq('XNA').sum() == placeholder_counts_before[
        'Type_Organization = XNA'
    ]

## 6. Investigate numeric sentinels and invalid ranges

A value can be numeric and still be invalid. Three issues require explicit rules:

- `Employed_Days = 365243` is far beyond a human working lifetime.
- the score variables are proportions/scores expected within 0 to 1.
- `Population_Region_Relative` is also expected within 0 to 1.

We inspect the suspicious employment value against income type before replacing it.

In [ ]:
employment_sentinel_mask = df['Employed_Days'].eq(365243)
employment_sentinel_by_income = pd.crosstab(
    df['Client_Income_Type'], employment_sentinel_mask
)
display(employment_sentinel_by_income)
print('Employment sentinel count:', int(employment_sentinel_mask.sum()))

range_violations_before = pd.Series({
    'Score_Source_1 outside [0, 1]': int(((~df['Score_Source_1'].between(0, 1)) & df['Score_Source_1'].notna()).sum()),
    'Score_Source_2 outside [0, 1]': int(((~df['Score_Source_2'].between(0, 1)) & df['Score_Source_2'].notna()).sum()),
    'Score_Source_3 outside [0, 1]': int(((~df['Score_Source_3'].between(0, 1)) & df['Score_Source_3'].notna()).sum()),
    'Population_Region_Relative outside [0, 1]': int(((~df['Population_Region_Relative'].between(0, 1)) & df['Population_Region_Relative'].notna()).sum()),
}, name='count')
display(range_violations_before.to_frame())

`365243` is almost entirely associated with retired applicants, confirming that it is a sentinel rather than a real duration. We create an explicit flag before replacing it with `NaN`; this preserves the distinction between this known sentinel and ordinary missingness. Six `Score_Source_2` values and two population-relative values equal 100, so they are also replaced with `NaN`.

In [ ]:
df['Employed_Days_365243_Flag'] = employment_sentinel_mask.astype('int8')
df.loc[employment_sentinel_mask, 'Employed_Days'] = np.nan

for column in ['Score_Source_1', 'Score_Source_2', 'Score_Source_3',
               'Population_Region_Relative']:
    invalid_range_mask = df[column].notna() & ~df[column].between(0, 1)
    df.loc[invalid_range_mask, column] = np.nan

assert df['Employed_Days'].eq(365243).sum() == 0
assert all(
    ((df[column].between(0, 1)) | df[column].isna()).all()
    for column in ['Score_Source_1', 'Score_Source_2', 'Score_Source_3',
                   'Population_Region_Relative']
)

## 7. Reassess high missingness instead of dropping columns automatically

The Data Understanding notebook proposed that `Own_House_Age` is mainly missing because some applicants do not own a house. We test that assumption directly.

In [ ]:
house_age_missing_by_ownership = (
    df.groupby('House_Own', dropna=False)['Own_House_Age']
      .apply(lambda values: values.isna().mean() * 100)
      .round(2)
      .rename('Own_House_Age missing (%)')
)
display(house_age_missing_by_ownership.to_frame())

missing_after_cleaning = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_percentage': (df.isna().mean() * 100).round(2),
}).sort_values('missing_percentage', ascending=False)

missing_after_cleaning[missing_after_cleaning['missing_count'] > 0].head(15)

`Own_House_Age` is missing for roughly 65% of both owners and non-owners. The earlier structural explanation is therefore insufficient. We retain this field and use median imputation with a missingness indicator. The same evidence-preserving approach is used for other numerical fields with missing values.

We also avoid deleting or winsorizing high-income and high-credit applications at this stage. Those observations can be genuine and may be important for risk prediction. Robust scaling will reduce their influence on scale-sensitive models without erasing them.

## 8. Define semantic feature roles

Pandas data types do not fully describe feature meaning. Several numeric codes are categories:

- binary ownership/contact indicators
- application day of week
- city rating
- the employment-sentinel flag

We encode these as categories instead of treating the difference between their numeric codes as a continuous measurement.

In [ ]:
numeric_categorical_features = [
    'Car_Owned', 'Bike_Owned', 'Active_Loan', 'House_Own',
    'Mobile_Tag', 'Homephone_Tag', 'Workphone_Working',
    'Cleint_City_Rating', 'Application_Process_Day',
    'Employed_Days_365243_Flag',
]

text_categorical_features = df.select_dtypes(include=['object', 'string']).columns.tolist()
categorical_features = text_categorical_features + numeric_categorical_features

excluded_columns = {'ID', 'Default'}
numeric_features = [
    column for column in df.columns
    if column not in excluded_columns and column not in categorical_features
]

# Convert semantic categories to a consistent string representation.
# This prevents numeric codes from being interpreted as continuous quantities.
for column in categorical_features:
    df[column] = df[column].astype('string')

feature_roles = pd.Series({
    'numeric_features': len(numeric_features),
    'categorical_features': len(categorical_features),
    'identifier_columns': 1,
    'target_columns': 1,
}, name='count')

display(feature_roles.to_frame())
print('Numeric features:', numeric_features)
print('Categorical features:', categorical_features)

assert len(numeric_features) + len(categorical_features) + 2 == df.shape[1]

## 9. Separate ID, predictors, and target

`ID` is retained separately for traceability but removed from `X`. Including it could encourage the model to learn accidental patterns from an arbitrary identifier.

In [ ]:
record_ids = df['ID'].copy()
X = df.drop(columns=['ID', 'Default'])
y = df['Default'].copy()

print('Predictor matrix:', X.shape)
print('Target vector:', y.shape)
print('ID retained separately:', record_ids.shape)

assert 'ID' not in X.columns
assert 'Default' not in X.columns

## 10. Create a stratified train/test split

We split before learning medians, category vocabularies, or scaling parameters. `stratify=y` keeps the default rate similar in both partitions. `random_state=42` makes the split reproducible.

The test set is held back for final evaluation; it must not guide preprocessing decisions or model tuning.

In [ ]:
X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
    X, y, record_ids,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

split_summary = pd.DataFrame({
    'rows': [len(X_train), len(X_test)],
    'default_count': [int(y_train.sum()), int(y_test.sum())],
    'default_percentage': [round(y_train.mean() * 100, 3), round(y_test.mean() * 100, 3)],
}, index=['train', 'test'])

display(split_summary)

assert set(X_train.index).isdisjoint(set(X_test.index))
assert len(X_train) + len(X_test) == len(X)
assert abs(y_train.mean() - y_test.mean()) < 0.001

## 11. Build the learned preprocessing pipeline

### Numerical features

- **Median imputation:** more robust than the mean for skewed income, credit, and annuity distributions.
- **Missing indicators:** preserve whether a value was absent instead of pretending the imputed median was observed.
- **Robust scaling:** centres by the median and scales using the interquartile range, reducing the influence of extreme but potentially valid applications.

### Categorical features

- **Explicit `Missing` category:** avoids inventing the most frequent category for an unknown applicant value.
- **One-hot encoding:** does not impose a false numeric order on categories.
- **Unknown-category handling:** future categories not seen in training produce an all-zero block rather than an error.

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
    ('scaler', RobustScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(
        strategy='constant', fill_value='Missing', missing_values=pd.NA
    )),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=True)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', numeric_pipeline, numeric_features),
        ('categorical', categorical_pipeline, categorical_features),
    ],
    remainder='drop',
    verbose_feature_names_out=True,
)

preprocessor

## 12. Fit on training data and transform both partitions

`fit_transform` is used only on `X_train`. The fitted medians, scale parameters, and category vocabulary are then reused by `transform(X_test)`. Calling `fit` on the test set would be data leakage.

In [ ]:
X_train_prepared = preprocessor.fit_transform(X_train)
X_test_prepared = preprocessor.transform(X_test)
prepared_feature_names = preprocessor.get_feature_names_out()

print('Prepared training shape:', X_train_prepared.shape)
print('Prepared test shape:', X_test_prepared.shape)
print('Prepared feature count:', len(prepared_feature_names))
print('First 15 feature names:')
display(pd.Series(prepared_feature_names[:15], name='feature_name').to_frame())

## 13. Validate the prepared matrices

A successful transformation is not enough. We verify row preservation, equal feature structure, and the absence of non-finite numerical values. For sparse matrices, only stored values need to be checked because unstored entries represent zero.

In [ ]:
train_stored_values = (
    X_train_prepared.data
    if hasattr(X_train_prepared, 'data')
    else np.asarray(X_train_prepared)
)
test_stored_values = (
    X_test_prepared.data
    if hasattr(X_test_prepared, 'data')
    else np.asarray(X_test_prepared)
)

validation_checks = pd.Series({
    'training_rows_preserved': X_train_prepared.shape[0] == len(X_train),
    'test_rows_preserved': X_test_prepared.shape[0] == len(X_test),
    'same_feature_count': X_train_prepared.shape[1] == X_test_prepared.shape[1],
    'training_values_finite': bool(np.isfinite(train_stored_values).all()),
    'test_values_finite': bool(np.isfinite(test_stored_values).all()),
    'target_alignment_train': X_train.index.equals(y_train.index),
    'target_alignment_test': X_test.index.equals(y_test.index),
}, name='passed')

display(validation_checks.to_frame())
assert validation_checks.all(), 'At least one preprocessing validation failed'

## 14. Save the cleaned split for the next notebook

We save the deterministically cleaned, **unencoded** partitions with their IDs and targets. Feature engineering can therefore continue separately on training and test data.

We do not save the fitted preprocessing object as the final modelling object yet. During cross-validation, preprocessing should be placed inside each model pipeline so it is refitted within every training fold.

In [ ]:
processed_dir = project_root / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

train_clean = pd.concat([
    id_train.rename('ID'), X_train, y_train.rename('Default')
], axis=1).reset_index(drop=True)

test_clean = pd.concat([
    id_test.rename('ID'), X_test, y_test.rename('Default')
], axis=1).reset_index(drop=True)

train_output_path = processed_dir / 'train_clean.csv'
test_output_path = processed_dir / 'test_clean.csv'

train_clean.to_csv(train_output_path, index=False)
test_clean.to_csv(test_output_path, index=False)

print(f'Saved training data: {train_output_path} | shape={train_clean.shape}')
print(f'Saved test data:     {test_output_path} | shape={test_clean.shape}')

## 15. Final preprocessing decisions

| Issue | Decision | Reason |
|---|---|---|
| Numeric corruption symbols | Coerce to `NaN` | Symbols are invalid numeric measurements |
| `Client_Gender = XNA` and `Accompany_Client = ##` | Convert to missing | Rare unknown/corrupt category codes |
| `Type_Organization = XNA` | Retain as its own category | Common and systematic, but its exact meaning is not verified |
| `Employed_Days = 365243` | Add flag, then convert to missing | Impossible duration and documented sentinel pattern |
| Scores or regional ratio outside 0–1 | Convert to missing | Values violate valid range |
| Duplicate rows | No action | None were found |
| `ID` | Retain separately; exclude from predictors | Traceability without identifier leakage |
| Missing numerical values | Training median plus indicator | Robust and preserves missingness signal |
| Missing categorical values | Explicit `Missing` category | Avoids inventing the modal category |
| Categorical variables | One-hot encode | Avoids false ordering |
| Numerical scale | Robust scaling | Limits influence of extreme but potentially valid values |
| Class imbalance | Stratified split only at this stage | Resampling/class weights belong inside model training |
| Outliers | Retain | No evidence that plausible extremes are erroneous |

### Deliberately postponed

- Feature engineering such as age in years, employment ratios, or credit-to-income ratios belongs in `04_feature_engineering.ipynb`.
- SMOTE or other sampling must be applied only to training folds and will be compared with class-weighted models.
- Model-specific preprocessing will be wrapped with each estimator during baseline modelling and cross-validation.